<a href="https://colab.research.google.com/github/rhigazi/pmTool/blob/main/PMTool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!npm install -g @opencode/cli

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 5 packages in 23s
⠦npm notice
npm notice New major version of npm available! 10.8.2 -> 12.2.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.2.2
npm notice To update run: npm install -g npm@12.2.2
npm notice
⠦

In [6]:
!curl --proto '=https' --tlsv1.2 -fsSL \
  https://github.com/lexmount/moli/releases/latest/download/moli-installer.sh | sh

Installed moli to /root/.local/bin/moli
Add /root/.local/bin to PATH, then run: moli --version


In [2]:
# @title Zelle 6: Telegram-Benachrichtigung (Abschluss)
import datetime
import os
import sys

import requests

# --- Laufzeit-Check & Secrets (Dual-Environment wie in Zelle 1/2) ---
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # python-dotenv fehlt (z.B. in Colab); Secrets kommen dann aus google.colab.userdata
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN")
    TELEGRAM_CHAT_ID = userdata.get("TELEGRAM_CHAT_ID")
else:
    TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
    TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")

if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
    print("Telegram-Secrets fehlen (TELEGRAM_BOT_TOKEN / TELEGRAM_CHAT_ID) – Benachrichtigung übersprungen.")
else:
    # --- Kurze Zusammenfassung für die Nachricht ---
    WORKDIR = os.environ.get("NEWSCHECK_DIR") or ("/content/newscheck" if IN_COLAB else os.getcwd())
    _okf_dir = os.path.join(WORKDIR, "okf-regeln")
    _okf_count = len([f for f in os.listdir(_okf_dir) if f.endswith(".md")]) if os.path.isdir(_okf_dir) else 0
    _erwartung = "erstellt/aktualisiert" if os.path.exists(os.path.join(WORKDIR, "erwartung.md")) else "fehlt"
    _rows = len(df) if "df" in globals() else "?"
    _stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

    text = f"""✅ Newscheck-Pipeline abgeschlossen ({_stamp})

• Marktdaten (DIA): {_rows} Zeilen
• OKF-Regeln: {_okf_count} Dateien in okf-regeln/
• erwartung.md: {_erwartung}
• Push nach origin/main: siehe vorherige Zelle"""

    # --- Telegram senden (best effort; Versandfehler sind nicht fatal) ---
    _url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    try:
        _resp = requests.post(_url, data={"chat_id": TELEGRAM_CHAT_ID, "text": text}, timeout=30)
        if _resp.status_code == 200 and _resp.json().get("ok"):
            print("Telegram-Nachricht gesendet ✓")
        else:
            print("Telegram-Versand fehlgeschlagen:", _resp.status_code, _resp.text)
    except requests.RequestException as e:
        print("Telegram-Versand fehlgeschlagen (Netzwerkfehler):", e)


Telegram-Nachricht gesendet ✓


In [3]:
# @title Telegram-Secrets → /content/config.json (für telegram_send.py / Agent)
import json
import sys
from pathlib import Path

# --- Nur in Colab sinnvoll ---
IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("Diese Zelle ist für Google Colab gedacht (google.colab.userdata).")

from google.colab import userdata

# --- Secrets lesen ---
bot_token = userdata.get("TELEGRAM_BOT_TOKEN")
chat_id = userdata.get("TELEGRAM_CHAT_ID")

# --- Validierung ---
missing = [k for k, v in {
    "TELEGRAM_BOT_TOKEN": bot_token,
    "TELEGRAM_CHAT_ID": chat_id,
}.items() if not v]
if missing:
    raise ValueError(f"Fehlende Colab-Secrets: {', '.join(missing)}")

# chat_id als String speichern (telegram_send.py akzeptiert String/Zahl)
config = {
    "bot_token": str(bot_token).strip(),
    "chat_id": str(chat_id).strip(),
}

# --- Nach /content schreiben (Projektroot, von telegram_send.py erwartet) ---
target = Path("/content/config.json")
target.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

# --- Kurze Bestätigung ohne Secrets preiszugeben ---
def _mask(s: str, keep: int = 4) -> str:
    s = str(s)
    return s[:keep] + "…" if len(s) > keep else "…"

print(f"✓ Konfiguration geschrieben: {target}")
print(f"  bot_token: {_mask(config['bot_token'])} (gesetzt)")
print(f"  chat_id:   {_mask(config['chat_id'])} (gesetzt)")
print("Hinweis: /content/config.json steht in .gitignore und wird nicht committet.")

✓ Konfiguration geschrieben: /content/config.json
  bot_token: 8686… (gesetzt)
  chat_id:   7482… (gesetzt)
Hinweis: /content/config.json steht in .gitignore und wird nicht committet.


In [4]:
from google.colab import userdata
import os

# 1. Token aus den Colab Secrets auslesen
github_token = userdata.get('GITHUB_TOKEN')

# 2. Git-Identität konfigurieren (bitte anpassen)
!git config --global user.name "rhigazi"
!git config --global user.email "rhigazi@gmail.com"

# 3. Repository klonen (Authentifizierung erfolgt über den Token in der URL)
repo_url = f"https://{github_token}@github.com/rhigazi/pmTool.git"

if not os.path.exists("pmTool"):
    !git clone {repo_url}
    print("Erfolgreich authentifiziert und Repository geklont!")
else:
    print("Repository 'pmTool' existiert bereits lokal.")

Cloning into 'pmTool'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 206 (delta 64), reused 186 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 179.47 KiB | 3.20 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Erfolgreich authentifiziert und Repository geklont!


In [5]:
!git clone https://github.com/rhigazi/pmTool && cd pmTool
!bash start.sh --colab          # Umgebung + App starten
# Telegram-Secrets-Zelle aus docs/colab-start.md (config.json wird nicht mitgeklont!)

fatal: destination path 'pmTool' already exists and is not an empty directory.
bash: start.sh: No such file or directory


Du bist der Projektmanager des Projekts "OKF Projekt-Manager (SPA)" gemäß AGENTS.md in diesem
Verzeichnis. Die vorherige Sitzung wurde geplant angehalten (Sitzungswechsel, Colab-Container neu).
Nicht neu anfangen — an den vorhandenen Stand anknüpfen.

Lies zuerst in dieser Reihenfolge (Session-Start-Regel):
1. docs/uebergabeprotokoll.md — Gesamtstand, Aufgabenstatus A1–A18, offene Punkte, Weiterführung
2. wissen/index.md und wissen/entscheidungen/log.md — Projektstand und getroffene Entscheidungen
3. Bei Bedarf docs/colab-start.md — Umgebungs-Wiederherstellung (bash start.sh --colab,
   Telegram-Secrets-Zelle; config.json ist gitignored!)

Kurz zum Stand: MVP (SPA zur OKF-Projektverwaltung) ist in Umsetzung. A1–A8, A10, A11, A16 sind
umgesetzt (Reviews teils ohne kritische Befunde, A10-Testphase bestanden), A9/A12/A15 waren beim
Halt in Arbeit (Zwischenstände stehen im Übergabeprotokoll), A13/A14/A17 sind offen. Letzter Stand
ist auf GitHub (origin/main) gesichert; Gesamtsuite zuletzt ~391 Tests grün.

Beginne so:
1. Umgebung prüfen: bash start.sh bzw. bash definitions/verify.sh . (Exit 0 = alles da).
2. Stand prüfen (Git-Status, offene Punkte im Übergabeprotokoll) und mir knapp den vorgeschlagenen
   Ablauf zeigen.
3. Dann weiterarbeiten gemäß Übergabeprotokoll, Abschnitt "Weiterführung": Zwischenstände klären →
   A18 abschließen → offene Aufgaben (A13/A14/A17) planen/vergeben → gebündelte Testphase
   (A1, A3–A8, A11, A16) → Doku-Runde → Commits je Takt → MVP-Abnahme.

Regeln (unverändert, siehe AGENTS.md und wissen/entscheidungen/log.md):
- Workflow (PM + Subagenten planner/coder/reviewer/tester/documenter) bleibt unverändert;
  Umsetzung ausschließlich über die Subagenten.
- App-Schreibrechte: index.md/features.md/log.md sind KI-Dateien (App generell gesperrt);
  Statuswechsel Klarung/Geschlossen und owner darf die Person, Pipeline-Status nur die KI.
- OKF-Pflege: jede Entscheidung ins wissen/entscheidungen/log.md, jede Umsetzung in
  wissen/features.md und wissen/log.md.
- Testisolation: Tests laufen gegen Test-Bundles (tmp_path, /tmp/opencode/…), nie gegen wissen/.
- Keine neuen Abhängigkeiten ohne Bescheid. UI-Tests laufen derzeit per HTTP + Node-DOM-Harness
  (Moli-Frage ist offen).
- Regelmäßig Sicherungs-Commits pushen (nach Takten und bei großen Statusmeldungen) —
  Container können ausfallen.
- Telegram (python3 tools/telegram_send.py --config <pfad>/config.json "…") bei Statusmeldungen,
  erledigten Aufgaben und bei Fragen an die Projektleitung; bei offenen Fragen mit
  Standardentscheidung weiterarbeiten.

Antworte auf Deutsch, halte mich über Fortschritt per Telegram auf dem Laufenden.

In [ ]:
import time
import psutil
from google.colab import output

# Schwellenwert in Prozent: Wenn die CPU-Auslastung darunter fällt, schlägt der Alarm
CPU_THRESHOLD = 10.0
CHECK_INTERVAL_SECONDS = 30  # Alle 30 Sekunden prüfen

print("🔄 CPU-Überwachung und Session-Keeper gestartet...")
print(f"Alarmiert, wenn CPU unter {CPU_THRESHOLD}% fällt.")

try:
    while True:
        # Aktuelle CPU-Auslastung ermitteln (über 1 Sekunde gemessen)
        cpu_usage = psutil.cpu_percent(interval=1)

        # Optional: Auch prüfen, ob das Notebook noch aktiv ist (verhindert Timeout)
        # print(f"Aktuelle CPU-Auslastung: {cpu_usage}%", end="\r")

        if cpu_usage < CPU_THRESHOLD:
            print(f"\n⚠️ Achtung! CPU-Auslastung ist niedrig ({cpu_usage}%). Aufgabe möglicherweise beendet!")

            # JavaScript-Audio-Piepton im Browser abspielen und Browser-Notification senden
            output.eval_js('''
                async function playBeep() {
                    const audioCtx = new (window.AudioContext || window.webkitAudioContext)();
                    const oscillator = audioCtx.createOscillator();
                    const gainNode = audioCtx.createGain();

                    oscillator.type = 'sine';
                    oscillator.frequency.setValueAtTime(800, audioCtx.currentTime); // Frequenz in Hz
                    gainNode.gain.setValueAtTime(0.5, audioCtx.currentTime);

                    oscillator.connect(gainNode);
                    gainNode.connect(audioCtx.destination);

                    oscillator.start();
                    setTimeout(() => {
                        oscillator.stop();
                        audioCtx.close();
                    }, 500); // 500 ms lang piepen
                }
                playBeep();
            ''')

            # Warte etwas länger, damit der Alarm nicht im Sekundentakt ertönt,
            # falls die Aufgabe dauerhaft beendet ist.
            time.sleep(120)

        # Intervall bis zur nächsten Prüfung
        time.sleep(CHECK_INTERVAL_SECONDS)

except KeyboardInterrupt:
    print("\n🛑 Überwachung manuell gestoppt.")

🔄 CPU-Überwachung und Session-Keeper gestartet...
Alarmiert, wenn CPU unter 10.0% fällt.

⚠️ Achtung! CPU-Auslastung ist niedrig (3.0%). Aufgabe möglicherweise beendet!

⚠️ Achtung! CPU-Auslastung ist niedrig (4.0%). Aufgabe möglicherweise beendet!

⚠️ Achtung! CPU-Auslastung ist niedrig (3.5%). Aufgabe möglicherweise beendet!

⚠️ Achtung! CPU-Auslastung ist niedrig (8.0%). Aufgabe möglicherweise beendet!

⚠️ Achtung! CPU-Auslastung ist niedrig (4.5%). Aufgabe möglicherweise beendet!

⚠️ Achtung! CPU-Auslastung ist niedrig (4.5%). Aufgabe möglicherweise beendet!

⚠️ Achtung! CPU-Auslastung ist niedrig (6.5%). Aufgabe möglicherweise beendet!

⚠️ Achtung! CPU-Auslastung ist niedrig (5.6%). Aufgabe möglicherweise beendet!

⚠️ Achtung! CPU-Auslastung ist niedrig (8.0%). Aufgabe möglicherweise beendet!

⚠️ Achtung! CPU-Auslastung ist niedrig (8.5%). Aufgabe möglicherweise beendet!

⚠️ Achtung! CPU-Auslastung ist niedrig (7.5%). Aufgabe möglicherweise beendet!

⚠️ Achtung! CPU-Auslastung ist